# 08 — Audit causal de la récupération physique

Ce carnet lit uniquement les essais de développement déjà enregistrés. Il ne lance ni entraînement ni simulation. Les figures et le lecteur privés se trouvent dans CGR_CAUSAL_ROOT/evidence. La qualification physique globale est une référence hors ligne, jamais une observation opérationnelle.

Trois configurations représentent deux familles géométriques. Les rendez-vous temporels d'un même épisode sont corrélés. La cohorte nominale fixe permet de suivre les mêmes véhicules, tandis que REC-EVAL-1 conserve sa population A0 à la première action.

In [ ]:
import os, json
from pathlib import Path
from IPython.display import display, Image
from cav_recovery.recovery.inspection import ConstructiveStudy
root = Path(os.environ['CGR_CAUSAL_ROOT'])
evidence = root / 'evidence'
data = json.loads((evidence / 'RESULTATS_CONSOLIDES.json').read_text('utf-8'))
check = ConstructiveStudy(root).verify()
assert not check['errors'], check['errors'][:5]
print('Manifestes :', check['checked'], 'fichiers vérifiés ;', len(data['results']), 'exécutions natives.')

## 1. Influence réelle et comparaison appariée

Le pilote historique est Graph-PPO, sans TCN opérationnel. La baseline coopérative ne lit pas les statuts D01 ou C-RDG pour choisir ses options. A et B diffèrent par l'information du décideur, avec le même garde local. C change aussi la planification et l'activation ; D change plusieurs paramètres de coordination. E et F corrigent le modèle de prévision sans changer le garde d'exécution.

In [ ]:
display(Image(filename=str(evidence/'figures/10_influence_effective.png'),width=850))
display(Image(filename=str(evidence/'figures/01_arrivees_cohortes.png'),width=900))
for case in ['oblique-4','oblique-8','crossing-4']:
    print(case, 'cohorte nominale :', data['nominal_cohorts'][case])
    for r in data['results']:
        if r['case'] == case:
            print(f"  {r['condition']:>14} : {r['arrivals']}/{r['introduced']} arrivées ; cohorte {len(r['cohort_arrived'])}/{len(data['nominal_cohorts'][case])} ; action {r['t_act_s']} ; {r['outcome']}")

## 2. Détection et communications

Un bon score d'alerte sur l'annotation physique ne démontre pas une intervention utile. Les comptes sont des rendez-vous corrélés, et le graphe local n'a pas l'obligation de reconstruire toute la scène.

In [ ]:
display(Image(filename=str(evidence/'figures/02_chronologie_detection.png'),width=850))
for case in ['oblique-4','oblique-8','crossing-4']:
    print(case, 'détection TP/FN/FP :', {mode:tuple(data['detector'][case]['counts'][mode].get(k,0) for k in ('tp','fn','fp')) for mode in ('raw','metro','tcn')})
    print('Graphe TP/FN/FP :',tuple(data['graph'][case]['counts'].get(k,0) for k in ('valid_tp','valid_fn','valid_fp')))
transport=json.loads((root/'report-transport-ablation.json').read_text('utf-8'))
for case in ('oblique-4','oblique-8'):
    print(case, 'un saut / deux sauts / agrégation idéale de rapports produits :',
          [transport[case]['variants'][mode]['valid_tp'] for mode in ('one_hop','two_hop','all_reports_ideal')])

## 3. Action, géométrie et sécurité

Les erreurs de prévision utilisent la distance réellement parcourue ; elles ne prédisent ni le temps ni la sûreté universelle. Le budget corrigé est contrôlé à chaque pas de physique de 0,02 s. La limite de 8 m reste inchangée.

In [ ]:
display(Image(filename=str(evidence/'figures/03_calibration_mouvements.png'),width=850))
display(Image(filename=str(evidence/'figures/04_budget_physique.png'),width=650))
calibration=json.loads((root/'motion-calibration.json').read_text('utf-8'))['results']
print('Essais :',len(calibration),' ; erreur maximale ancienne / corrigée :',
      max(r['legacy_position_error_m'] for r in calibration),
      max(r['center_position_error_m'] for r in calibration))

## 4. Trajectoires natives et résultat défavorable

À quatre CAV, D termine à 129,0 s au lieu de 165,2 s pour A. À huit CAV, aucun bras ne libère les quatre membres de la cohorte nominale avant 180 s. Dans la scène de traversée, le nominal se résout seul, alors que le déclenchement TCN suivi du protocole actuel laisse quatre véhicules bloqués. Ce constat concerne cette combinaison et ce scénario apparié.

In [ ]:
for name in ['05_recuperation_locale_quatre','06_echec_global_huit','07_intervention_tcn_defavorable','09_chronologie_intervention_inutile']:
    display(Image(filename=str(evidence/'figures'/f'{name}.png'),width=850))
print('Lecteur temporel :',evidence/'LECTEUR_SEQUENCES.html')

## 5. Conclusion contrôlable

Le rapport privé RAPPORT_AUDIT_CAUSAL.md détaille les causes, les facteurs changés, les verdicts et le prochain critère de passage. Le tableau complet et les métadonnées des dix figures sont conservés dans evidence/. Aucune généralisation sur le test indépendant n'est établie ici.